# FastMCP: The Framework for MCP

FastMCP is the standard framework for building Model Context Protocol (MCP) servers, clients, and interactive applications.

**FastMCP is a full framework for building Model Context Protocol (MCP) applications**. It gives you one coherent API for servers, clients, and interactive apps. Use it to expose Python functions as MCP tools, connect to local or remote MCP servers, and return interactive interfaces directly from your tools. FastMCP manages schema generation, validation, transport, authentication, and protocol compatibility around your application code.

In [1]:
%%writefile "servers/1.py"
from fastmcp import FastMCP

mcp = FastMCP("Server")


@mcp.tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


if __name__ == "__main__":
    mcp.run()

Overwriting servers/1.py


In [2]:
import subprocess
import os

script_path = os.path.join("servers", "1.py")

process = subprocess.Popen(
    ["python", script_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"Server started with PID: {process.pid}")

Server started with PID: 13915


In [3]:
stdout, stderr = process.communicate(timeout=1)
print(stdout)
print(stderr)




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.0.2                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      Server, 4.0.2                               │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                         

In [4]:
process.terminate()

if process.poll() is not None:
    print("Server stopped successfully.")
else:
    print("Server still running, forcing kill...")
    process.kill()

Server stopped successfully.


An effective MCP application needs more than a function registry. Models need accurate schemas, callers need validated results, clients need compatible transports, and production servers need authentication and predictable lifecycle management.

FastMCP treats those as framework responsibilities. Declare a Python function and FastMCP derives its schema, validates its inputs and outputs, and exposes it through MCP. Connect a client to a URL and FastMCP handles protocol negotiation, authentication, and connection lifecycle. Your application remains ordinary Python while FastMCP keeps the MCP boundary correct.

That’s why FastMCP is the standard framework for working with MCP. FastMCP created the high-level Python API incorporated into the official MCP Python SDK in 2024. The actively maintained standalone project is now downloaded more than a million times a day, and some version of FastMCP powers 70% of MCP servers across all languages.

FastMCP three pillars:
- **Servers** turn your application logic into MCP capabilities with generated schemas and validation.
- **Clients** connect to local or remote MCP servers with full protocol support.
- **Apps** let tools return forms, tables, charts, and other interactive interfaces alongside ordinary MCP results.

## Installation

It is recommended to using uv to install and manage FastMCP.

```bash
uv add fastmcp
```

```bash
pip install fastmcp
```

FastMCP provides optional extras for specific features. For example, to install the background tasks extra:

```bash
pip install "fastmcp[tasks]"
```

In [5]:
!fastmcp version

FastMCP version:                                                           4.0.2
MCP version:                                                               2.1.1
Python version:                                                          3.12.13
Platform:                             Linux-6.1.0-52-amd64-x86_64-with-glibc2.36
FastMCP root path: /home/aliakbar/miniconda3/envs/llm/lib/python3.12/site-packa…


## Quick Start

A FastMCP server is a collection of tools, resources, and other MCP components. To create a server, start by instantiating the FastMCP class.

```python
from fastmcp import FastMCP

mcp = FastMCP("My MCP Server")
```

To add a tool that returns a simple greeting, write a function and decorate it with @mcp.tool to register it with the server:

In [7]:
%%writefile "servers/2.py"
from fastmcp import FastMCP

mcp = FastMCP("My MCP Server")

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}!"

Overwriting servers/2.py


to run the server

In [1]:
%%writefile "servers/2.py"
from fastmcp import FastMCP

mcp = FastMCP("My MCP Server")

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Overwriting servers/2.py


In [2]:
import subprocess
import os

script_path = os.path.join("./servers", "2.py")

process = subprocess.Popen(
    ["python", script_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"Server started with PID: {process.pid}")

Server started with PID: 15479


In [3]:
stdout, stderr = process.communicate(timeout=1)
print(stdout)
print(stderr)




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 4.0.2                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      My MCP Server, 4.0.2                        │
│                  🚀 Deploy free: https://horizon.prefect.io                  │
│                         

In [4]:
process.terminate()

if process.poll() is not None:
    print("Server stopped successfully.")
else:
    print("Server still running, forcing kill...")
    process.kill()

Server stopped successfully.


the last version uses stdio to handling io of mcp servers. To run a http server and access to it on http protocol use this `trnasport="http"` and `port=[PORT]` in mcp.run method as arguments.

In [10]:
%%writefile "servers/3.py"
from fastmcp import FastMCP

mcp = FastMCP("My MCP Server")

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="http", port=8002)


Overwriting servers/3.py


run int your terminal
```bash
python servers/3.py
```

### Call Your Server as Client

Once your server is running with HTTP transport or stdio, you can connect to it with a FastMCP client or any LLM client that supports the MCP protocol:



In [18]:
%%writefile "clients/1.py"
import asyncio
from fastmcp import Client

client = Client("http://localhost:8002/mcp")
async def call_tool(name: str):
    async with client:
        result = await client.call_tool("greet", {"name": name})
        print(result)
asyncio.run(call_tool("Ali"))

Overwriting clients/1.py


In [19]:
import subprocess
import os

script_path = os.path.join("./clients", "1.py")

process = subprocess.Popen(
    ["python", script_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(f"Server started with PID: {process.pid}")

Server started with PID: 19149


In [21]:
stdout, stderr = process.communicate(timeout=1)
print(stdout)
print(stderr)

CallToolResult(content=[TextContent(type='text', text='Hello, Ali!', annotations=None, meta=None)], structured_content={'result': 'Hello, Ali!'}, meta={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'My MCP Server', 'version': '4.0.2'}}, data='Hello, Ali!', is_error=False)




In [22]:
process.terminate()

if process.poll() is not None:
    print("Server stopped successfully.")
else:
    print("Server still running, forcing kill...")
    process.kill()

Server stopped successfully.


FastMCP clients are asynchronous, so the call goes through asyncio.run. Entering the client context with async with client: is what opens the connection, and it stays open for as many calls as you want to make inside the block.

## Fast MCP Server In depth

The `FastMCP` class is the central piece of every FastMCP application. It acts as the container for your tools, resources, and prompts, managing communication with MCP clients and orchestrating the entire server lifecycle.

### Creating Server

At its simplest, a FastMCP server just needs a name. Everything else has sensible defaults.

```python
from fastmcp import FastMCP

mcp = FastMCP("MyServer")
```

Instructions help clients (and the LLMs behind them) understand what your server does and how to use it effectively.

```python
mcp = FastMCP(
    "DataAnalysis",
    instructions="Provides tools for analyzing numerical datasets. Start with get_summary() for an overview.",
)
```

### Components

FastMCP servers expose three types of components to clients, each serving a distinct role in the MCP protocol.
- **Tools** are functions that clients invoke to perform actions or access external systems.
- **Resources** expose data that clients can read — passive data sources rather than invocable functions.
- **Prompts** are reusable message templates that guide LLM interactions.

**Tools**

```python
@mcp.tool
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers together."""
    return a * b
```

**Resources**

```python
@mcp.resource("data://config")
def get_config() -> dict:
    return {"theme": "dark", "version": "1.0"}
```

**Prompts**
```python
@mcp.prompt
def analyze_data(data_points: list[float]) -> str:
    formatted_data = ", ".join(str(point) for point in data_points)
    return f"Please analyze these data points: {formatted_data}"
```

### Running the Server

FastMCP supports several transports:

- STDIO (default): For local integrations and CLI tools
- HTTP: For web services using the Streamable HTTP protocol
- SSE: Legacy web transport (deprecated)


```python
from fastmcp import FastMCP

mcp = FastMCP("MyServer")

@mcp.tool
def greet(name: str) -> str:
    """Greet a user by name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()
```

HTTP Mode:

```python
mcp.run(transport="http", host="127.0.0.1", port=9000)
```

The FastMCP constructor accepts parameters organized into four categories: identity, composition, behavior, and handlers.

#### Identity

These parameters control how your server presents itself to clients.

- `name`: str | None default:"None"<br>
A human-readable name for your server, shown in client applications and logs. If omitted, FastMCP generates a random name
- `instructions`: str | None<br>
Description of how to interact with this server. Clients surface these instructions to help LLMs understand the server’s purpose and available functionality

- `version`:str | int | float | None<br>
Version string for your server. Defaults to the FastMCP library version if not provided

#### Composition

These parameters control what your server is built from — its components, middleware, providers, and lifecycle.

- `tools`: Sequence[Tool | Callable] | None<br>
Tools to register on the server. An alternative to the @mcp.tool decorator when you need to add tools programmatically
- `auth`: AuthProvider | None<br>
Authentication provider for securing HTTP-based transports. See Authentication for configuration
- `middleware`: Sequence[Middleware] | None<br>
Middleware that intercepts and transforms every MCP message flowing through the server — requests, responses, and notifications in both directions. Use for cross-cutting concerns like logging, error handling, and rate limiting
- `providers`: Sequence[Provider] | None<br>
Providers that supply tools, resources, and prompts dynamically. Providers are queried at request time, so they can serve components from databases, APIs, or other external sources
- `lifespan`: Lifespan | LifespanCallable | None<br>
Server-level setup and teardown logic that runs when the server starts and stops.

#### Behavior

These parameters tune how the server processes requests and communicates with clients.

- `on_duplicate`: Literal["warn", "error", "replace", "ignore"]default:"warn"<br>
How to handle duplicate component registrations

#### Response Caching

A server whose listings and resource reads change slowly can tell clients how long they may reuse a response before fetching it again (SEP-2549). Set cache_ttl (seconds) on the server, and the hint is attached uniformly to every cacheable response — tools/list, prompts/list, resources/list, resources/templates/list, and resources/read.

```python
from fastmcp import FastMCP

mcp = FastMCP("Weather", cache_ttl=300, cache_scope="public")

@mcp.tool
def forecast(city: str) -> str:
    return f"Sunny in {city}"
```

`cache_scope` controls whether a cached response may be shared across authorization contexts ("public") or reused only within the one that produced it ("private", the default when a TTL is set). A `cache_scope` without a `cache_ttl` does not enable caching and raises at construction.

### Tag Based Filtering

Tags let you categorize components and selectively expose them. This is useful for creating different views of your server for different environments or user types.

```python
@mcp.tool(tags={"public", "utility"})
def public_tool() -> str:
    return "This tool is public"

@mcp.tool(tags={"internal", "admin"})
def admin_tool() -> str:
    return "This tool is for admins only"
```

The filtering logic works as follows:

- Enable with only=True: Switches to allowlist mode — only components with at least one matching tag are exposed
- Disable: Components with any matching tag are hidden
- Precedence: Later calls override earlier ones, so call disable after enable to exclude from an allowlist


```python
# Only expose components tagged with "public"
mcp = FastMCP()
mcp.enable(tags={"public"}, only=True)

# Hide components tagged as "internal" or "deprecated"
mcp = FastMCP()
mcp.disable(tags={"internal", "deprecated"})

# Combine both: show admin tools but hide deprecated ones
mcp = FastMCP()
mcp.enable(tags={"admin"}, only=True).disable(tags={"deprecated"})
```

## Tools

Tools are the core building blocks that allow your LLM to interact with external systems, execute code, and access data that isn’t in its training data. In FastMCP, tools are Python functions exposed to LLMs through the MCP protocol.

Tools in FastMCP transform regular Python functions into capabilities that LLMs can invoke during conversations. When an LLM decides to use a tool:

1. It sends a request with parameters based on the tool’s schema.
2. FastMCP validates these parameters against your function’s signature.
3. Your function executes with the validated inputs.
4. The result is returned to the LLM, which can use it in its response.

This allows LLMs to perform tasks like querying databases, calling APIs, making calculations, or accessing files—extending their capabilities beyond what’s in their training data.

Creating a tool is as simple as decorating a Python function with @mcp.tool:

```python
from fastmcp import FastMCP

mcp = FastMCP(name="CalculatorServer")

@mcp.tool
def add(a: int, b: int) -> int:
    """Adds two integer numbers together."""
    return a + b
```

#### Decorator Arguments

```python
@mcp.tool(
    name="find_products",           # Custom tool name for the LLM
    description="Search the product catalog with optional category filtering.", # Custom description
    tags={"catalog", "search"},      # Optional tags for organization/filtering
    meta={"version": "1.2", "author": "product-team"}  # Custom metadata
)
def search_products_implementation(query: str, category: str | None = None) -> list[dict]:
    """Internal function description (ignored if description is provided above)."""
    # Implementation...
    print(f"Searching for '{query}' in category '{category}'")
    return [{"id": 2, "name": "Another Product"}]
```

using class methods as a mcp tools:

```python
from fastmcp import FastMCP
from fastmcp.tools import tool

class Calculator:
    def __init__(self, multiplier: int):
        self.multiplier = multiplier

    @tool()
    def multiply(self, x: int) -> int:
        """Multiply x by the instance multiplier."""
        return x * self.multiplier

calc = Calculator(multiplier=3)
mcp = FastMCP()
mcp.add_tool(calc.multiply)  # Registers with correct schema (only 'x', not 'self')
```

#### Arguments

By default, FastMCP converts Python functions into MCP tools by inspecting the function’s signature and type annotations. This allows you to use standard Python type annotations for your tools. In general, the framework strives to “just work”: idiomatic Python behaviors like parameter defaults and type annotations are automatically translated into MCP schemas. However, there are a number of ways to customize the behavior of your tools.

MCP tools have typed arguments, and FastMCP uses type annotations to determine those types. Therefore, you should use standard Python type annotations for tool arguments:

```python
@mcp.tool
def analyze_text(
    text: str,
    max_tokens: int = 100,
    language: str | None = None
) -> dict:
    """Analyze the provided text."""
    # Implementation...
```

FastMCP provides helper classes for returning images, audio, and files. When you return one of these classes, either directly or as part of a list, FastMCP automatically converts it to the appropriate MCP content block. For example, if you return a fastmcp.utilities.types.Image object, FastMCP will convert it to an MCP ImageContent block with the correct MIME type and base64 encoding.

```python
from fastmcp.utilities.types import Image, Audio, File

@mcp.tool
def get_chart() -> Image:
    """Generate a chart image."""
    return Image(path="chart.png")

@mcp.tool
def get_multiple_charts() -> list[Image]:
    """Return multiple charts."""
    return [Image(path="chart1.png"), Image(path="chart2.png")]
```

## MCP Annotations

FastMCP allows you to add specialized metadata to your tools through annotations. These annotations communicate how tools behave to client applications without consuming token context in LLM prompts.

Annotations serve several purposes in client applications:

- Adding user-friendly titles for display purposes
- Indicating whether tools modify data or systems
- Describing the safety profile of tools (destructive vs. non-destructive)
- Signaling if tools interact with external systems

```python
from mcp.types import ToolAnnotations

@mcp.tool(
    annotations=ToolAnnotations(
        title="Calculate Sum",
        readOnlyHint=True,
        openWorldHint=False,
    )
)
def calculate_sum(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b
```

An example

```python
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("Data Server")

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True))
def get_user(user_id: str) -> dict:
    """Retrieve user information by ID."""
    return {"id": user_id, "name": "Alice"}

@mcp.tool(
    annotations=ToolAnnotations(
        readOnlyHint=True,
        idempotentHint=True,  # Same result for repeated calls
        openWorldHint=False   # Only internal data
    )
)
def search_products(query: str) -> list[dict]:
    """Search the product catalog."""
    return [{"id": 1, "name": "Widget", "price": 29.99}]

# Write operations - no readOnlyHint
@mcp.tool()
def update_user(user_id: str, name: str) -> dict:
    """Update user information."""
    return {"id": user_id, "name": name, "updated": True}

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def delete_user(user_id: str) -> dict:
    """Permanently delete a user account."""
    return {"deleted": user_id}
```

### MCP Context

Tools can access MCP features like logging, reading resources, or reporting progress through the Context object. To use it, add a parameter to your tool function with the type hint Context.

```python
from fastmcp import FastMCP, Context

mcp = FastMCP(name="ContextDemo")

@mcp.tool
async def process_data(data_uri: str, ctx: Context) -> dict:
    """Process data from a resource with progress reporting."""
    await ctx.info(f"Processing data from {data_uri}")

    result = await ctx.read_resource(data_uri)
    data = result.contents[0].content if result.contents else ""
    await ctx.report_progress(progress=50, total=100)

    summary = str(data)[:200]
    await ctx.report_progress(progress=100, total=100)
    return {"length": len(data), "summary": summary}
```

The Context object provides access to:

- Logging: ctx.debug(), ctx.info(), ctx.warning(), ctx.error()
-Progress Reporting: ctx.report_progress(progress, total)
- Resource Access: ctx.read_resource(uri)
- Request Information: ctx.request_id, ctx.client_id

### Removing Tools

```python
from fastmcp import FastMCP

mcp = FastMCP(name="DynamicToolServer")

@mcp.tool
def calculate_sum(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

mcp.local_provider.remove_tool("calculate_sum")
```

## Resources & Templates

Resources represent data or files that an MCP client can read, and resource templates extend this concept by allowing clients to request dynamically generated resources based on parameters passed in the URI.

What Are Resources?
Resources provide read-only access to data for the LLM or client application. When a client requests a resource URI:

- FastMCP finds the corresponding resource definition.
- If it’s dynamic (defined by a function), the function is executed.
- The content (text, JSON, binary data) is returned to the client.

This allows LLMs to access files, database content, configuration, or dynamically generated information relevant to the conversation.

### `@resource` Decorator

```python
import json
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Basic dynamic resource returning a string
@mcp.resource("resource://greeting")
def get_greeting() -> str:
    """Provides a simple greeting message."""
    return "Hello from FastMCP Resources!"

# Resource returning JSON data
@mcp.resource("data://config")
def get_config() -> str:
    """Provides application configuration as JSON."""
    return json.dumps({
        "theme": "dark",
        "version": "1.2.0",
        "features": ["tools", "resources"],
    })
```

- `URI`: The first argument to @resource is the unique URI (e.g., "resource://greeting") clients use to request this data.
- `Lazy Loading`: The decorated function (get_greeting, get_config) is only executed when a client specifically requests that resource URI via resources/read.

```python
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Example specifying metadata
@mcp.resource(
    uri="data://app-status",      # Explicit URI (required)
    name="ApplicationStatus",     # Custom name
    description="Provides the current status of the application.", # Custom description
    mime_type="application/json", # Explicit MIME type
    tags={"monitoring", "status"}, # Categorization tags
    meta={"version": "2.1", "team": "infrastructure"}  # Custom metadata
)
def get_application_status() -> str:
    """Internal function description (ignored if description is provided above)."""
    return json.dumps({"status": "ok", "uptime": 12345, "version": "2.1"})
```

To return a custom type object

```python
from fastmcp import FastMCP
from fastmcp.resources import ResourceResult, ResourceContent

mcp = FastMCP()

@mcp.resource("data://users")
def get_users() -> ResourceResult:
    return ResourceResult(
        contents=[
            ResourceContent(content='[{"id": 1}]', mime_type="application/json"),
            ResourceContent(content="# Users\n...", mime_type="text/markdown"),
        ],
        meta={"total": 1}
    )
```

You can control which resources are enabled for clients using server-level enabled control. Disabled resources don’t appear in list_resources and can’t be read.

```python
from fastmcp import FastMCP

mcp = FastMCP("MyServer")

@mcp.resource("data://public", tags={"public"})
def get_public(): return "public"

@mcp.resource("data://secret", tags={"internal"})
def get_secret(): return "secret"

# Disable specific resources by key
mcp.disable(names={"data://secret"})

# Disable resources by tag
mcp.disable(tags={"internal"})

# Or use allowlist mode - only enable resources with specific tags
mcp.enable(tags={"public"}, only=True)
```

### Query parameters

```python 
from fastmcp import FastMCP

mcp = FastMCP(name="DataServer")

# Basic query parameters
@mcp.resource("data://{id}{?format}")
def get_data(id: str, format: str = "json") -> str:
    """Retrieve data in specified format."""
    if format == "xml":
        return f"<data id='{id}' />"
    return f'{{"id": "{id}"}}'

# Multiple query parameters with type coercion
@mcp.resource("api://{endpoint}{?version,limit,offset}")
def call_api(endpoint: str, version: int = 1, limit: int = 10, offset: int = 0) -> dict:
    """Call API endpoint with pagination."""
    return {
        "endpoint": endpoint,
        "version": version,
        "limit": limit,
        "offset": offset,
        "results": fetch_results(endpoint, version, limit, offset)
    }

# Query parameters with wildcards
@mcp.resource("files://{path*}{?encoding,lines}")
def read_file(path: str, encoding: str = "utf-8", lines: int = 100) -> str:
    """Read file with optional encoding and line limit."""
    return read_file_content(path, encoding, lines)
```

## Prompts

Prompts are reusable message templates that help LLMs generate structured, purposeful responses. FastMCP simplifies defining these templates, primarily using the @mcp.prompt decorator.

`What Are Prompts?`


Prompts provide parameterized message templates for LLMs. When a client requests a prompt:

- FastMCP finds the corresponding prompt definition.
- If it has parameters, they are validated against your function signature.
- Your function executes with the validated inputs.
- The generated message(s) are returned to the LLM to guide its response.

This allows you to define consistent, reusable templates that LLMs can use across different clients and contexts.

```python
from fastmcp import FastMCP
from fastmcp.prompts import Message

mcp = FastMCP(name="PromptServer")

# Basic prompt returning a string (converted to user message automatically)
@mcp.prompt
def ask_about_topic(topic: str) -> str:
    """Generates a user message asking for an explanation of a topic."""
    return f"Can you please explain the concept of '{topic}'?"

# Prompt returning multiple messages
@mcp.prompt
def generate_code_request(language: str, task_description: str) -> list[Message]:
    """Generates a conversation for code generation."""
    return [
        Message(f"Write a {language} function that performs the following task: {task_description}"),
        Message("I'll help you write that function.", role="assistant"),
    ]
```

#### Decorator Arguments

```python
@mcp.prompt(
    name="analyze_data_request",          # Custom prompt name
    description="Creates a request to analyze data with specific parameters",  # Custom description
    tags={"analysis", "data"},            # Optional categorization tags
    meta={"version": "1.1", "author": "data-team"}  # Custom metadata
)
def data_analysis_prompt(
    data_uri: str = Field(description="The URI of the resource containing the data."),
    analysis_type: str = Field(default="summary", description="Type of analysis.")
) -> str:
    """This docstring is ignored when description is provided."""
    return f"Please perform a '{analysis_type}' analysis on the data found at {data_uri}."
```

Return Message type data

```python
from fastmcp.prompts import Message

@mcp.prompt
def roleplay_scenario(character: str, situation: str) -> list[Message]:
    """Sets up a roleplaying scenario with initial messages."""
    return [
        Message(f"Let's roleplay. You are {character}. The situation is: {situation}"),
        Message("Okay, I understand. I am ready. What happens next?", role="assistant")
    ]
```

PromptResult

```python
from fastmcp import FastMCP
from fastmcp.prompts import PromptResult, Message

mcp = FastMCP(name="PromptServer")

@mcp.prompt
def code_review(code: str) -> PromptResult:
    """Returns a code review prompt with metadata."""
    return PromptResult(
        messages=[
            Message(f"Please review this code:\n\n```\n{code}\n```"),
            Message("I'll analyze this code for issues.", role="assistant"),
        ],
        description="Code review prompt",
        meta={"review_type": "security", "priority": "high"}
    )
```

## Context

When defining FastMCP tools, resources, resource templates, or prompts, your functions might need to interact with the underlying MCP session or access advanced server capabilities. FastMCP provides the Context object for this purpose.

`What Is Context?`


The Context object provides a clean interface to access MCP features within your functions, including:

- Logging: Send debug, info, warning, and error messages back to the client
- Progress Reporting: Update the client on the progress of long-running operations
- Resource Access: List and read data from resources registered with the server
- Prompt Access: List and retrieve prompts registered with the server
- User Elicitation: Request structured input from users during tool execution
- Request State: Pass values and non-serializable resources between middleware and handlers within a request (for state that persists across requests, see Session State)
- Session Visibility: Control which components are visible to the current session
- Request Information: Access metadata about the current request
- Server Access: When needed, access the underlying FastMCP server instance


The preferred way to access context is using the CurrentContext() dependency:

```python
from fastmcp import FastMCP
from fastmcp.dependencies import CurrentContext
from fastmcp.server.context import Context

mcp = FastMCP(name="Context Demo")

@mcp.tool
async def process_file(file_uri: str, ctx: Context = CurrentContext()) -> str:
    """Processes a file, using context for logging and resource access."""
    await ctx.info(f"Processing {file_uri}")
    return "Processed file"
```

This works with tools, resources, and prompts:

```python
from fastmcp import FastMCP
from fastmcp.dependencies import CurrentContext
from fastmcp.server.context import Context

mcp = FastMCP(name="Context Demo")

@mcp.resource("resource://user-data")
async def get_user_data(ctx: Context = CurrentContext()) -> dict:
    await ctx.debug("Fetching user data")
    return {"user_id": "example"}

@mcp.prompt
async def data_analysis_request(dataset: str, ctx: Context = CurrentContext()) -> str:
    return f"Please analyze the following dataset: {dataset}"
```

### Logging methods

```python
await ctx.debug("Starting analysis")
await ctx.info(f"Processing {len(data)} items") 
await ctx.warning("Deprecated parameter used")
await ctx.error("Processing failed")
```

### Client Elicitation

```python
result = await ctx.elicit("Enter your name:", response_type=str)
if result.action == "accept":
    name = result.data
```

### Progress Report

```python
await ctx.report_progress(progress=50, total=100)  # 50% complete
```

### Request State

Request state carries values within a single request, across the middleware → handler pipeline. A request runs through any middleware you’ve added and then the handler — separate functions that don’t share a stack frame, so a plain local variable can’t pass anything between them. ctx.set_state / ctx.get_state is that channel.

The common case is a middleware that resolves something once and every tool reads it, rather than each tool recomputing it:

```python
from fastmcp import FastMCP, Context
from fastmcp.server.middleware import Middleware, MiddlewareContext

mcp = FastMCP("app")


class Enrich(Middleware):
    async def on_call_tool(self, context: MiddlewareContext, call_next):
        await context.fastmcp_context.set_state("caller", "alice")
        return await call_next(context)


mcp.add_middleware(Enrich())


@mcp.tool
async def whoami(ctx: Context) -> str:
    return await ctx.get_state("caller") or "unknown"
```

## Transforms

Transforms modify components as they flow from providers to clients. When a client asks “what tools do you have?”, the request passes through each transform in the chain. Each transform can modify the components before passing them along.

Think of transforms as filters in a pipeline. Components flow from providers through transforms to reach clients:

```
Provider → [Transform A] → [Transform B] → Client
```

When listing components, transforms receive sequences and return transformed sequences—a pure function pattern. When getting a specific component by name, transforms use a middleware pattern with call_next, working in reverse: mapping the client’s requested name back to the original, then transforming the result.

FastMCP provides several transforms for common use cases:

- Namespace - Prefix component names to prevent conflicts when composing servers
- Tool Transformation - Rename tools, modify descriptions, reshape arguments
- Enabled - Control which components are visible at runtime
- Tool Search - Replace large tool catalogs with on-demand search
- Resources as Tools - Expose resources to tool-only clients
- Prompts as Tools - Expose prompts to tool-only clients
- Code Mode (Experimental) - Replace many tools with programmable search + execute

Transforms can be added at two levels, each serving different purposes.

- Provider-Level Transforms
- Server-Level Transforms

### Provider-Level Transforms

Provider transforms apply to components from a specific provider. They run first, modifying components before they reach the server level.

```python
from fastmcp import FastMCP
from fastmcp.server.providers import FastMCPProvider
from fastmcp.server.transforms import Namespace, ToolTransform
from fastmcp.tools.tool_transform import ToolTransformConfig

sub_server = FastMCP("Sub")

@sub_server.tool
def process(data: str) -> str:
    return f"Processed: {data}"

# Create provider and add transforms
provider = FastMCPProvider(sub_server)
provider.add_transform(Namespace("api"))
provider.add_transform(ToolTransform({
    "api_process": ToolTransformConfig(description="Process data through the API"),
}))

main = FastMCP("Main", providers=[provider])
# Tool is now: api_process with updated description
```

### Server-Level Transforms

Server transforms apply to all components from all providers. They run after provider transforms, seeing the already-transformed names.

```python
from fastmcp import FastMCP
from fastmcp.server.transforms import Namespace

mcp = FastMCP("Server", transforms=[Namespace("v1")])

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}!"

# All tools become v1_toolname
```

### Custom Transforms

Create custom transforms by subclassing Transform and overriding the methods you need.

```python
from collections.abc import Sequence
from fastmcp.server.transforms import Transform, GetToolNext
from fastmcp.tools import Tool

class TagFilter(Transform):
    """Filter tools to only those with specific tags."""

    def __init__(self, required_tags: set[str]):
        self.required_tags = required_tags

    async def list_tools(self, tools: Sequence[Tool]) -> Sequence[Tool]:
        return [t for t in tools if t.tags & self.required_tags]

    async def get_tool(self, name: str, call_next: GetToolNext) -> Tool | None:
        tool = await call_next(name)
        if tool and tool.tags & self.required_tags:
            return tool
        return None
```

The Transform base class provides default implementations that pass through unchanged. Override only the methods relevant to your transform.

Each component type has two methods with different patterns:

List methods receive sequences directly and return transformed sequences. Get methods use call_next for routing flexibility—when a client requests “new_name”, your transform maps it back to “original_name” before calling call_next().

## Tool Transformation

Tool transformation lets you modify tool schemas - renaming tools, changing descriptions, adjusting tags, and reshaping argument schemas. FastMCP provides two mechanisms that share the same configuration options but differ in timing.

**Deferred transformation** with ToolTransform applies modifications when tools flow through a transform chain. Use this for tools from mounted servers, proxies, or other providers where you don’t control the source directly.

**Immediate transformation** with Tool.from_tool() creates a modified tool object right away. Use this when you have direct access to a tool and want to transform it before registration.

### ToolTransform

The ToolTransform class is a transform that modifies tools as they flow through a provider. Provide a dictionary mapping original tool names to their transformation configuration.

```python
from fastmcp import FastMCP
from fastmcp.server.transforms import ToolTransform
from fastmcp.tools.tool_transform import ToolTransformConfig

mcp = FastMCP("Server")

@mcp.tool
def verbose_internal_data_fetcher(query: str) -> str:
    """Fetches data from the internal database."""
    return f"Results for: {query}"

# Rename the tool to something simpler
mcp.add_transform(ToolTransform({
    "verbose_internal_data_fetcher": ToolTransformConfig(
        name="search",
        description="Search the database.",
    )
}))

# Clients see "search" with the cleaner description
```

ToolTransform is useful when you want to modify tools from mounted or proxied servers without changing the original source.

### Tool.from_tool()

Use Tool.from_tool() when you have the tool object and want to create a transformed version for registration.

```python
from fastmcp import FastMCP
from fastmcp.tools import Tool, tool
from fastmcp.tools.tool_transform import ArgTransform

# Create a tool without registering it
@tool
def search(q: str, limit: int = 10) -> list[str]:
    """Search for items."""
    return [f"Result {i} for {q}" for i in range(limit)]

# Transform it before registration
better_search = Tool.from_tool(
    search,
    name="find_items",
    description="Find items matching your search query.",
    transform_args={
        "q": ArgTransform(
            name="query",
            description="The search terms to look for.",
        ),
    },
)

mcp = FastMCP("Server")
mcp.add_tool(better_search)
```

## Search Tools

### Regex Search

```python
from fastmcp import FastMCP
from fastmcp.server.transforms.search import RegexSearchTransform

mcp = FastMCP("My Server", transforms=[RegexSearchTransform()])

@mcp.tool
def search_database(query: str, limit: int = 10) -> list[dict]:
    """Search the database for records matching the query."""
    ...

@mcp.tool
def delete_record(record_id: str) -> bool:
    """Delete a record from the database by its ID."""
    ...

@mcp.tool
def send_email(to: str, subject: str, body: str) -> bool:
    """Send an email to the given recipient."""
    ...
```

The LLM’s search_tools call takes a pattern parameter — a regex string:

```python 
# Exact substring match
result = await client.call_tool("search_tools", {"pattern": "database"})
# Returns: search_database, delete_record

# Regex pattern
result = await client.call_tool("search_tools", {"pattern": "send.*email|notify"})
# Returns: send_email
```

### Limiting Results

```python
mcp.add_transform(RegexSearchTransform(max_results=10))
mcp.add_transform(BM25SearchTransform(max_results=3))
```

### Pinning Tools

```python
mcp.add_transform(RegexSearchTransform(
    always_visible=["help", "status"],
))

# list_tools returns: help, status, search_tools, call_tool
```

### Custom Tool Names

```python
mcp.add_transform(RegexSearchTransform(
    search_tool_name="find_tools",
    call_tool_name="run_tool",
))
```

### The call_tool Proxy

The call_tool proxy forwards calls to the real tool. When a client calls call_tool(name="search_database", arguments={...}), the proxy resolves search_database through the server’s normal tool pipeline — including transforms and middleware — and executes it. The proxy rejects attempts to call the synthetic tools themselves. call_tool(name="call_tool") raises an error rather than recursing.

### Visibility

```python
from fastmcp.server.transforms import Visibility
from fastmcp.server.transforms.search import RegexSearchTransform

mcp = FastMCP("My Server")

# ... define tools ...

# Disable admin tools globally
mcp.add_transform(Visibility(False, tags={"admin"}))

# Add search — admin tools won't appear in results
mcp.add_transform(RegexSearchTransform())
```

## Providers

Every FastMCP server has one or more component providers. A provider is a source of tools, resources, and prompts - it’s what makes components available to clients.

When a client connects to your server and asks “what tools do you have?”, FastMCP asks each provider that question and combines the results. When a client calls a specific tool, FastMCP finds which provider has it and delegates the call. You’re already using providers. When you write @mcp.tool, you’re adding a tool to your server’s LocalProvider - the default provider that stores components you define directly in code. You just don’t have to think about it for simple servers. Providers become important when your components come from multiple sources: another FastMCP server to include, a remote MCP server to proxy, or a database where tools are defined dynamically. Each source gets its own provider, and FastMCP queries them all seamlessly.

The provider abstraction solves a common problem: as servers grow, you need to organize components across multiple sources without tangling everything together.
- **Composition**: Break a large server into focused modules. A “weather” server and a “calendar” server can each be developed independently, then mounted into a main server. Each mounted server becomes a FastMCPProvider.
- **Proxying**: Expose a remote MCP server through your local server. Maybe you’re bridging transports (remote HTTP to local stdio) or aggregating multiple backends. Remote connections become ProxyProvider instances.
- **Dynamic sources**: Load tools from a database, generate them from an OpenAPI spec, or create them based on user permissions. Custom providers let components come from anywhere.

FastMCP includes providers for common patterns:

| Provider | What It Does | How You Use It |
| :--- | :--- | :--- |
| **LocalProvider** | Stores components you define in code | `@mcp.tool`, `mcp.add_tool()` |
| **FastMCPProvider** | Wraps another FastMCP server | `mcp.mount(server)` |
| **ProxyProvider** | Connects to remote MCP servers | `create_proxy(client)` |

### Filesystem Provider

FileSystemProvider scans a directory for Python files and automatically registers functions decorated with @tool, @resource, or @prompt. This enables a file-based organization pattern similar to Next.js routing, where your project structure becomes your component registry.

Create a provider pointing to your components directory, then pass it to your server. Use Path(__file__).parent to make the path relative to your server file.

```python
from pathlib import Path

from fastmcp import FastMCP
from fastmcp.server.providers import FileSystemProvider

mcp = FastMCP("MyServer", providers=[FileSystemProvider(Path(__file__).parent / "components")])
```

In your components/ directory, create Python files with decorated functions.

```python
# components/tools/greet.py
from fastmcp.tools import tool

@tool
def greet(name: str) -> str:
    """Greet someone by name."""
    return f"Hello, {name}!"
```

When the server starts, FileSystemProvider scans the directory, imports all Python files, and registers any decorated functions it finds.

```
components/
├── tools/
│   ├── greeting.py      # @tool functions
│   └── calculator.py    # @tool functions
├── resources/
│   └── config.py        # @resource functions
└── prompts/
    └── assistant.py     # @prompt func
```

## MCP Proxy Provider

The Proxy Provider sources components from another MCP server through a client connection. This lets you expose any MCP server’s tools, resources, and prompts through your own server, whether the source is local or accessed over the network.

The Proxy Provider enables:

- Bridge transports: Make an HTTP server available via stdio, or vice versa
- Aggregate servers: Combine multiple source servers into one unified server
- Add security: Act as a controlled gateway with authentication and authorization
- Simplify access: Provide a stable endpoint even if backend servers change


```python
from fastmcp.server import create_proxy

# create_proxy() accepts URLs, file paths, and transports directly
proxy = create_proxy("http://example.com/mcp", name="MyProxy")

if __name__ == "__main__":
    proxy.run()
```

### Transport Bridging

```python
from fastmcp.server import create_proxy

# Bridge HTTP server to local stdio
http_proxy = create_proxy("http://example.com/mcp/sse", name="HTTP-to-stdio")

# Run locally via stdio for Claude Desktop
if __name__ == "__main__":
    http_proxy.run()  # Defaults to stdio
```

```python
from pathlib import Path
from fastmcp.server import create_proxy

# Bridge local server to HTTP
local_proxy = create_proxy(Path("local_server.py"), name="stdio-to-HTTP")

if __name__ == "__main__":
    local_proxy.run(transport="http", host="0.0.0.0", port=8080)
```

## Composition Servers

As your application grows, you’ll want to split it into focused servers — one for weather, one for calendar, one for admin — and combine them into a single server that clients connect to. That’s what mount() does.

When you mount a server, all its tools, resources, and prompts become available through the parent. The connection is live: add a tool to the child after mounting, and it’s immediately visible through the parent.

```python
from fastmcp import FastMCP

weather = FastMCP("Weather")

@weather.tool
def get_forecast(city: str) -> str:
    """Get weather forecast for a city."""
    return f"Sunny in {city}"

@weather.resource("data://cities")
def list_cities() -> list[str]:
    """List supported cities."""
    return ["London", "Paris", "Tokyo"]

main = FastMCP("MainApp")
main.mount(weather)

# main now serves get_forecast and data://cities
```

### Mounting External Servers

```python
from pathlib import Path
from fastmcp import FastMCP
from fastmcp.server import create_proxy

mcp = FastMCP("Orchestrator")

# Mount a remote HTTP server (URLs work directly)
mcp.mount(create_proxy("http://api.example.com/mcp"), namespace="api")

# Mount local Python scripts (file paths work directly)
mcp.mount(create_proxy(Path("./my_server.py")), namespace="local")
```

```python
weather = FastMCP("Weather")
calendar = FastMCP("Calendar")

@weather.tool
def get_data() -> str:
    return "Weather data"

@calendar.tool
def get_data() -> str:
    return "Calendar data"

main = FastMCP("Main")
main.mount(weather, namespace="weather")
main.mount(calendar, namespace="calendar")

# Tools are now:
# - weather_get_data
# - calendar_get_data

```

## User Elicitation

Ask users for input while a tool is running, on both the handshake and modern protocols.

User elicitation allows MCP servers to request input from users during tool execution. Instead of requiring all inputs upfront, tools can interactively ask for missing parameters, clarification, or additional context as needed.

Elicitation enables tools to request specific information from users mid-task:

- Missing parameters: Ask for required information not provided initially
- Clarification requests: Get user confirmation or choices for ambiguous scenarios
- Progressive disclosure: Collect complex information step-by-step
- Dynamic workflows: Adapt tool behavior based on user responses

User elicitation allows MCP servers to request input from users during tool execution. Instead of requiring all inputs upfront, tools can interactively ask for missing parameters, clarification, or additional context as needed.

Elicitation enables tools to request specific information from users mid-task:

- Missing parameters: Ask for required information not provided initially
- Clarification requests: Get user confirmation or choices for ambiguous scenarios
- Progressive disclosure: Collect complex information step-by-step
- Dynamic workflows: Adapt tool behavior based on user responses

For example, a file management tool might ask “Which directory should I create?” or a data analysis tool might request “What date range should I analyze?”

Use the ctx.elicit() method within any tool function to request user input on a handshake-era connection. Specify the message to display and the type of response you expect.

```python
from fastmcp import FastMCP, Context
from dataclasses import dataclass

mcp = FastMCP("Elicitation Server")

@dataclass
class UserInfo:
    name: str
    age: int

@mcp.tool
async def collect_user_info(ctx: Context) -> str:
    """Collect user information through interactive prompts."""
    result = await ctx.elicit(
        message="Please provide your information",
        response_type=UserInfo
    )

    if result.action == "accept":
        user = result.data
        return f"Hello {user.name}, you are {user.age} years old"
    elif result.action == "decline":
        return "Information not provided"
    else:  # cancel
        return "Operation cancelled"
```

The elicitation result contains an action field indicating how the user responded:

Action|Description
------|-----------
accept|User provided valid input—data is available in the data field
decline| User chose not to provide the requested information
cancel| User cancelled the entire operation

## Icon Defenition

### Server Icons

```python
from fastmcp import FastMCP
from mcp.types import Icon

mcp = FastMCP(
    name="WeatherService",
    website_url="https://weather.example.com",
    icons=[
        Icon(
            src="https://weather.example.com/icon-48.png",
            mime_type="image/png",
            sizes=["48x48"]
        ),
        Icon(
            src="https://weather.example.com/icon-96.png",
            mime_type="image/png",
            sizes=["96x96"]
        ),
    ]
)
```

### Tools Icons

```python
from mcp.types import Icon

@mcp.tool(
    icons=[Icon(src="https://example.com/calculator-icon.png")]
)
def calculate_sum(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b
```

```python
@mcp.prompt(
    icons=[Icon(src="https://example.com/prompt-icon.png")]
)
def analyze_code(code: str):
    """Create a prompt for code analysis."""
    return f"Please analyze this code:\n\n{code}"
```

## Middleware

Middleware adds behavior that applies across multiple operations—authentication, logging, rate limiting, or request transformation—without modifying individual tools or resources.

MCP middleware forms a pipeline around your server’s operations. When a request arrives, it flows through each middleware in order—each can inspect, modify, or reject the request before passing it along. After the operation completes, the response flows back through the same middleware in reverse order.

```
Request → Middleware A → Middleware B → Handler → Middleware B → Middleware A → Response
```

This bidirectional flow means middleware can:

- Pre-process: Validate authentication, log incoming requests, check rate limits
- Post-process: Transform responses, record timing metrics, handle errors consistently


The key decision point is call_next(context). Calling it continues the chain; not calling it stops processing entirely.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware import Middleware, MiddlewareContext

class LoggingMiddleware(Middleware):
    async def on_message(self, context: MiddlewareContext, call_next):
        print(f"→ {context.method}")
        result = await call_next(context)
        print(f"← {context.method}")
        return result

mcp = FastMCP("MyServer")
mcp.add_middleware(LoggingMiddleware())
```

Middleware executes in the order added to the server. The first middleware runs first on the way in and last on the way out:

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.error_handling import ErrorHandlingMiddleware
from fastmcp.server.middleware.rate_limiting import RateLimitingMiddleware
from fastmcp.server.middleware.logging import LoggingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(ErrorHandlingMiddleware())   # 1st in, last out
mcp.add_middleware(RateLimitingMiddleware())    # 2nd in, 2nd out
mcp.add_middleware(LoggingMiddleware())         # 3rd in, first out
```

When using mounted servers, middleware behavior follows a clear hierarchy:

- Parent middleware runs for all requests, including those routed to mounted servers
- Mounted server middleware only runs for requests handled by that specific server


### Hooks

Rather than processing every message identically, FastMCP provides specialized hooks at different levels of specificity. Multiple hooks fire for a single request, going from general to specific:

Level|Hooks|Purpose
-----|-----|-------
Message|on_message|All MCP traffic (requests and notifications)
Type|on_request, on_notification|Requests expecting responses vs fire-and-forget
Operation|on_call_tool, on_read_resource, on_get_prompt, etc.|Specific MCP operations

`on_message`: Called for every MCP message—both requests and notifications.

```python
async def on_message(self, context: MiddlewareContext, call_next):
    result = await call_next(context)
    return result
```

`on_request`: Called for MCP requests that expect a response.

```python
async def on_request(self, context: MiddlewareContext, call_next):
    result = await call_next(context)
    return result
```

`on_notification`: Called for fire-and-forget MCP notifications.

```python
async def on_notification(self, context: MiddlewareContext, call_next):
    await call_next(context)
    # Notifications don't return values
```

`on_call_tool`: Called when a tool is executed. The context.message contains name (tool name) and arguments (dict).

```python
async def on_call_tool(self, context: MiddlewareContext, call_next):
    tool_name = context.message.name
    args = context.message.arguments
    result = await call_next(context)
    return result
```

`on_read_resource`: Called when a resource is read. The context.message contains uri (resource URI).

```python
async def on_read_resource(self, context: MiddlewareContext, call_next):
    uri = context.message.uri
    result = await call_next(context)
    return result
```

`on_get_prompt`: Called when a prompt is retrieved. The context.message contains name (prompt name) and arguments (dict).

```python
async def on_get_prompt(self, context: MiddlewareContext, call_next):
    prompt_name = context.message.name
    result = await call_next(context)
    return result
```

`on_list_tools`: Called when listing available tools. Returns a list of FastMCP Tool objects before MCP conversion.

```python
async def on_list_tools(self, context: MiddlewareContext, call_next):
    tools = await call_next(context)
    # Filter or modify the tool list
    return tools
```

`on_list_resources`: Called when listing available resources. Returns FastMCP Resource objects.

```python
async def on_list_resources(self, context: MiddlewareContext, call_next):
    resources = await call_next(context)
    return resources
```

`on_list_resource_templates`: Called when listing resource templates.

```python
async def on_list_resource_templates(self, context: MiddlewareContext, call_next):
    templates = await call_next(context)
    return templates
```

### Built-in Middleware

`LoggingMiddleware` provides human-readable request and response logging.

`StructuredLoggingMiddleware` outputs JSON-formatted logs for aggregation tools like Datadog or Splunk.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.logging import LoggingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(LoggingMiddleware(
    include_payloads=True,
    max_payload_length=1000
))
```

`TimingMiddleware` logs execution duration for all requests.

`DetailedTimingMiddleware` provides per-operation timing with separate tracking for tools, resources, and prompts.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.timing import TimingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(TimingMiddleware())
```

Caches tool calls, resource reads, and list operations with TTL-based expiration.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.caching import ResponseCachingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(ResponseCachingMiddleware())
```

`RateLimitingMiddleware` uses a token bucket algorithm allowing controlled bursts.

`SlidingWindowRateLimitingMiddleware` provides precise time-window rate limiting without burst allowance.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.rate_limiting import RateLimitingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(RateLimitingMiddleware(
    max_requests_per_second=10.0,
    burst_capacity=20
))
```

```python
from fastmcp.server.middleware.rate_limiting import SlidingWindowRateLimitingMiddleware

mcp.add_middleware(SlidingWindowRateLimitingMiddleware(
    max_requests=100,
    window_minutes=1
))
```

`ErrorHandlingMiddleware` provides centralized error logging and transformation.

`RetryMiddleware` automatically retries with exponential backoff for transient failures.

```python
from fastmcp import FastMCP
from fastmcp.server.middleware.error_handling import ErrorHandlingMiddleware

mcp = FastMCP("MyServer")
mcp.add_middleware(ErrorHandlingMiddleware(
    include_traceback=True,
    transform_errors=True,
    error_callback=my_error_callback
))
```

## Lifespans

Lifespans let you run code once when the server starts and clean up when it stops. Unlike per-session handlers, lifespans run exactly once regardless of how many clients connect.

```python
from fastmcp import FastMCP
from fastmcp.server.lifespan import lifespan

@lifespan
async def app_lifespan(server):
    # Setup: runs once when server starts
    print("Starting up...")
    try:
        yield {"started_at": "2024-01-01"}
    finally:
        # Teardown: runs when server stops
        print("Shutting down...")

mcp = FastMCP("MyServer", lifespan=app_lifespan)
```

Access the lifespan context in tools via ctx.lifespan_context:

```python
from fastmcp import FastMCP, Context
from fastmcp.server.lifespan import lifespan

@lifespan
async def app_lifespan(server):
    # Initialize shared state
    data = {"users": ["alice", "bob"]}
    yield {"data": data}

mcp = FastMCP("MyServer", lifespan=app_lifespan)

@mcp.tool
def list_users(ctx: Context) -> list[str]:
    data = ctx.lifespan_context["data"]
    return data["users"]
```

Compose multiple lifespans with the | operator:

```python
from fastmcp import FastMCP
from fastmcp.server.lifespan import lifespan

@lifespan
async def config_lifespan(server):
    config = {"debug": True, "version": "1.0"}
    yield {"config": config}

@lifespan
async def data_lifespan(server):
    data = {"items": []}
    yield {"data": data}

# Compose with |
mcp = FastMCP("MyServer", lifespan=config_lifespan | data_lifespan)
```

# MCP Clients

The fastmcp.Client class provides a programmatic interface for interacting with any MCP server. It handles protocol details and connection management automatically, letting you focus on the operations you want to perform.

The FastMCP Client is designed for deterministic, controlled interactions rather than autonomous behavior, making it ideal for testing MCP servers during development, building deterministic applications that need reliable MCP interactions, and creating the foundation for agentic or LLM-based clients with structured, type-safe operations.

You provide a server source and the client automatically infers the appropriate transport mechanism.

```python
from pathlib import Path
import asyncio
from fastmcp import Client, FastMCP

# In-memory server (ideal for testing)
server = FastMCP("TestServer")
client = Client(server)

# HTTP server
client = Client("https://example.com/mcp")

# Local Python script
client = Client(Path("my_mcp_server.py"))

async def main():
    async with client:
        # List available operations
        tools = await client.list_tools()
        resources = await client.list_resources()
        prompts = await client.list_prompts()

        # Execute operations
        result = await client.call_tool("example_tool", {"param": "value"})
        print(result)

asyncio.run(main())
```

### Configuration-Based Clients

```python
config = {
    "mcpServers": {
        "weather": {
            "url": "https://weather-api.example.com/mcp"
        },
        "assistant": {
            "command": "python",
            "args": ["./assistant_server.py"]
        }
    }
}

client = Client(config)

async with client:
    # Tools are prefixed with server names
    weather_data = await client.call_tool("weather_get_forecast", {"city": "London"})
    response = await client.call_tool("assistant_answer_question", {"question": "What's the capital of France?"})

    # Resources use prefixed URIs
    icons = await client.read_resource("weather://weather/icons/sunny")
```

### Operations

Tools are server-side functions that the client can execute with arguments. Call them with call_tool() and receive structured results.

```python
async with client:
    tools = await client.list_tools()
    result = await client.call_tool("multiply", {"a": 5, "b": 3})
    print(result.data)  # 15
```

Resources are data sources that the client can read, either static or templated. Access them with read_resource() using URIs.

```python
async with client:
    resources = await client.list_resources()
    content = await client.read_resource("file:///config/settings.json")
    print(content[0].text)
```

Prompts are reusable message templates that can accept arguments. Retrieve rendered prompts with get_prompt().

```python
async with client:
    prompts = await client.list_prompts()
    messages = await client.get_prompt("analyze_data", {"data": [1, 2, 3]})
    print(messages.messages)
```